In [ ]:
"""
adding notagen corpus
"""

from music21 import *
notagencorpus = corpus.corpora.LocalCorpus("notagen")
notagencorpus.addPath(r"~\notagen-pieces\scores\Baroque_BachJohannSebastian_Choral")
notagencorpus.save()
notagencorpus.all()[0].parse().show("text")

In [ ]:
"""
1. Keys:
Algorithms: determine key of chorale by applying music21's key finding algorithm.
- count how often each key occurs, e.g.
bach_keys = { "C": 23, "C#": 1. "D": 12, ..."c": 32, "c#": 2, ...}
notagen_keys = { "C": 58, "C#": 1. "D": 6, ..."c": 14, "c#": 2, ...}

2. Time signatures:
Algorithm: take first time signature
bach_tss = { "4/4": 23, "3/4": 1. "5/8": 0, ...}
notagen_tss = { "4/4": 58, "3/4": 1. "5/8": 1, ...}

3. Melodies:
Algorithm:
  - transpose everything to C major / a minor (Bach AND Notagen)
  - take only 10 first notes of Soprano

4. Dominant seventh chords
...

5. Parallel Fiths (Google search)

6. Dv / Neapolitaner
...

7. Auftakte / Volltakte
"""

In [ ]:
"""
J:
1. Keys:
Algorithms: determine key of chorale by applying music21's key finding algorithm.
- count how often each key occurs, e.g.
bach_keys = { "C": 23, "C#": 1. "D": 12, ..."c": 32, "c#": 2, ...}
notagen_keys = { "C": 58, "C#": 1. "D": 6, ..."c": 14, "c#": 2, ...}
"""
import music21 as m21
from collections import Counter
keys_BACH = []
keys_notagen = []


bach_chorales = m21.corpus.chorales.Iterator()
for c in bach_chorales:
    chorales_key = c.analyze("key")
    keys_BACH.append(f"{chorales_key.tonic.name} {chorales_key.mode}")
keys_total = Counter(keys_BACH)
sorted(keys_total.items()) # this casts keys_total to a list!

# notagen_pieces = m21.converter.parse(notagen_corpus)
# for piece in notagen_pieces:
#   ai_keys = piece.analyze("key")
#   keys_notagen.append(f"{piece.tonic.name} {piece.mode}")

# keys_ai_total = Counter(keys_notagen)
# sorted(keys_ai_total.items())


# How do we iterate over the notagen corpus? This Code is not working

[('A major', 32),
 ('A minor', 45),
 ('A- major', 1),
 ('B major', 1),
 ('B minor', 28),
 ('B- major', 25),
 ('B- minor', 2),
 ('C major', 17),
 ('C minor', 8),
 ('D major', 25),
 ('D minor', 26),
 ('E major', 8),
 ('E minor', 15),
 ('E- major', 10),
 ('F major', 24),
 ('F minor', 2),
 ('F# minor', 6),
 ('G major', 52),
 ('G minor', 44)]

In [ ]:
"""
2. Time signatures:
Algorithm: take first time signature
bach_tss = { "4/4": 23, "3/4": 1. "5/8": 0, ...}
notagen_tss = { "4/4": 58, "3/4": 1. "5/8": 1, ...}
"""
""" NOTAGEN CHORALES; Iteration doesn't work
from collections import Counter
notagen_tss_list = []
notagen_chorales = m21.corpus.corpora.LocalCorpus("ai-bach").Iterator() #?
for piece in notagen_chorales:
  time_sig = piece.recurse().getElementsByClass(m21.meter.TimeSignature)[0]
  notagen_tss_list.append(time_sig.ratioString)
notagen_tss = Counter(notagen_tss_list)
print(notagen_tss)
"""
from collections import Counter

bach_tss_list = []
bach_chorales = m21.corpus.chorales.Iterator()
for piece in bach_chorales:
    time_sig = piece.recurse().getElementsByClass(m21.meter.TimeSignature)[0]
    bach_tss_list.append(time_sig.ratioString)
bach_tss = Counter(bach_tss_list)
print(bach_tss)


# Notagen Corpus must still be done! Bach Corpus works.

In [ ]:
"""
3. Melodies:
Algorithm:
  - transpose everything to C major / a minor (Bach AND Notagen)
  - take only 10 first notes of Soprano
"""

def extract_snippets(corpus=None):
  if corpus is not None:
    snippets = {}
    for chorale in corpus:
      try:
        ## transpose to C
        key = chorale.analyze("key")
        if key.mode == "major":
          interval = m21.interval.Interval(key.tonic, m21.pitch.Pitch("C"))
        elif key.mode == "minor":
          interval = m21.interval.Interval(key.tonic, m21.pitch.Pitch("A"))
        else:
          print("error")

        transposed = chorale.transpose(interval)
        for part in chorale.parts:
          if part.partName == "Soprano": # does that work? We should do smell-tests
            id = transposed.metadata.movementName
            soprano = transposed.flatten().notes  
            notes = [note.pitch.nameWithOctave for note in soprano.notes][:10]
            snippets[id] = notes
          else:
            continue
      except:
        pass

    return snippets
  else:
    return None

bach_chorales = m21.corpus.chorales.Iterator()
bach_snippets = extract_snippets(corpus=bach_chorales)

print(bach_snippets)

In [ ]:
import music21 as m21
snippets = {}
bach_chorales = m21.corpus.chorales.Iterator()
for piece in bach_chorales:
    for part in piece.parts:
        if part.partName == "Soprano":
            id = piece.metadata.movementName
            soprano = part.flatten().notes
            notes = [note.pitch.nameWithOctave for note in soprano.notes][:10]
            snippets[id] = notes
snippets

            


In [ ]:
"""
4. Dominant seventh chords
"""
d7_count = 0
bach_chorales = m21.corpus.chorales.Iterator()
for piece in bach_chorales:
    pieceChords = piece.chordify()
    for chord in pieceChords.recurse().getElementsByClass(m21.chord.Chord):
        if chord.isDominantSeventh():
            print(chord.measureNumber, chord.beatStr, chord)
            d7_count += 1
print(f"Total D7-Chord Count: {d7_count}")



6 3 1/2 <music21.chord.Chord D3 C4 F#4 A4>
9 2 1/2 <music21.chord.Chord A2 D4 F#4 C5>
13 3 <music21.chord.Chord G2 D4 F4 B4>
15 1 1/2 <music21.chord.Chord F#2 C4 A4 D5>
16 3 1/2 <music21.chord.Chord D3 C4 F#4 A4>
20 3 1/2 <music21.chord.Chord D3 C4 F#4 A4>
2 2 1/2 <music21.chord.Chord B2 A3 D#4 F#4>
4 2 1/2 <music21.chord.Chord B2 A3 D#4 F#4>
7 4 <music21.chord.Chord F#3 C#4 E4 A#4>
9 2 <music21.chord.Chord E3 G#3 D4 B4>
12 4 3/4 <music21.chord.Chord E3 D4 G#4 B4>
1 4 <music21.chord.Chord B3 D4 G#4 E5>
2 2 1/2 <music21.chord.Chord A3 D4 F#4 C5>
3 2 1/2 <music21.chord.Chord E3 D4 G#4 B4>
5 3 1/2 <music21.chord.Chord A3 B3 D#4 F#4>
2 2 1/2 <music21.chord.Chord E3 D4 G#4 B4>
3 4 1/2 <music21.chord.Chord E3 F#3 C#4 A#4>
4 2 1/2 <music21.chord.Chord F#3 A#3 E4 C#5>
1 2 1/2 <music21.chord.Chord D3 A3 F#4 C5>
4 2 <music21.chord.Chord D3 C4 F#4 A4>
6 2 1/2 <music21.chord.Chord D3 F#3 C4 A4>
7 1 1/2 <music21.chord.Chord F3 G3 D4 B4>
7 3 1/2 <music21.chord.Chord B2 G3 F4 D5>
10 4 1/2 <music21.ch